In [1]:
import numpy as np
import tensorflow as tf
import joblib 

2026-03-23 00:38:55.483350: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-23 00:38:55.483813: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-23 00:38:55.486593: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-23 00:38:55.494233: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-23 00:38:55.507725: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [2]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(float)
train_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_train_pred.npz')
train_targets_pred = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(float)
validation_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_validation_pred.npz')
validation_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(float)
test_targets = npz['outputs'].astype(int)

npz = np.load('Audiobooks_data_test_pred.npz')
y_pred_test = npz['outputs'].astype(int)




In [3]:
import pandas as pd

df = pd.DataFrame({
    'y_true': test_targets,
    'y_score': y_pred_test
})

df = df.sort_values(by='y_score', ascending=False)
df = df.reset_index(drop=True)

In [4]:

baseline = df['y_true'].mean() #out of all the users how many actually bought

def precision_at_k(df, k):
    cutoff = int(len(df) * k)
    top_k = df.iloc[:cutoff]
    return top_k['y_true'].mean()

k=0.1
p10 = precision_at_k(df, k)
p20 = precision_at_k(df, 0.2)
print("Precision@10%:", precision_at_k(df, p10))
print("Precision@20%:", precision_at_k(df, p20))

print(f"If baseline conversion = {(baseline*100).round(2)}% then Precision@10% = {(p10*100).round(2)}%")
print(f"If I target top {k*100}% users, {(p10*100).round(2)}% may actually convert.")

Precision@10%: 0.6101190476190477
Precision@20%: 0.59375
If baseline conversion = 51.12% then Precision@10% = 75.0%
If I target top 10.0% users, 75.0% may actually convert.


In [5]:
#lift

def lift_at_k(df, k):
    return precision_at_k(df, k) / baseline

top_user_percentage = 0.1
lift = lift_at_k(df, top_user_percentage).round(2)
print("Lift@10%:", lift)

print(f"The top {top_user_percentage * 100}% of users selected by the model are {lift}× more likely to convert than an average user.")

Lift@10%: 1.47
The top 10.0% of users selected by the model are 1.47× more likely to convert than an average user.
